# StockSense — smoke tests

End-to-end checks for the agentic pipeline: core screen, single-stock deep dive,
report rendering, and the MCP server (in-process client). Live cells hit yfinance;
run top-to-bottom. Defaults are rules-only (no LLM key needed).

In [ ]:
import asyncio
from pipeline_core.services import llm_service
from pipeline_core.config import settings

print('app:', settings.app_name, '| llm_mode:', llm_service.llm_mode(), '| index:', settings.default_index)

## 1. Screen NIFTY 50 (live, ~30s cold / seconds warm)

In [ ]:
from pipeline_core.agents.orchestrator import OrchestratorAgent
from pipeline_core.models.schemas import PipelineRequest, ScreenFilters

orch = OrchestratorAgent()
results = await orch.screen_only(PipelineRequest(
    filters=ScreenFilters(index='NIFTY 50', pe_ratio_max=25.0, roe_min=0.15),
    max_candidates=5, include_report=False,
))
for r in results:
    print(f'{r.ticker:12s} match={r.filter_match_pct:5.1f}%  PE={r.pe_ratio and round(r.pe_ratio,1)}  ROE={r.roe}%')
    print('   ', [(e.filter_name, e.passed, e.actual_value) for e in r.filter_explanations])

## 2. Deep dive on one ticker (4 agents on a shared StockContext)

In [ ]:
analysis = await orch.analyze_ticker('INFY', 'standard')
print('composite:', analysis['composite_score'])
for k in ('fundamental_score', 'technical_score', 'sentiment_score', 'risk_score'):
    print(f'  {k}:', analysis[k])
print('signal:', analysis['technical_signal'], '| risk:', analysis['risk_level'], '| roce:', analysis['roce'])
print('fundamental strengths:', analysis['evidence']['fundamental'].get('strengths'))
print('risk flags:', analysis['evidence']['risk'].get('risk_flags'))

## 3. Report rendering (template-first, works with no LLM)

In [ ]:
from pipeline_core.agents.report_generator import ReportGeneratorAgent
from IPython.display import Markdown, display

md = await ReportGeneratorAgent().generate_markdown_report(
    symbol='INFY', name=analysis.get('name', 'Infosys'),
    fundamental=analysis['evidence']['fundamental'],
    technical=analysis['evidence']['technical'],
    sentiment=analysis['evidence']['sentiment'],
    risk=analysis['evidence']['risk'],
    composite_score=analysis['composite_score'],
)
display(Markdown(md))

## 4. MCP server in-process (tools + resources, no network)

In [ ]:
from fastmcp import Client
from server import mcp

async with Client(mcp) as client:
    tools = await client.list_tools()
    print('tools:', sorted(t.name for t in tools))
    print('health:', (await client.call_tool('health', {})).data)
    await client.call_tool('save_screen', {
        'name': 'nb-quality', 'description': 'notebook demo',
        'filters': {'roe_min': 0.15, 'debt_to_equity_max': 0.5}})
    print('screens:', (await client.call_tool('list_saved_screens', {})).data['screens'])
    print('delete:', (await client.call_tool('delete_saved_screen', {'name': 'nb-quality'})).data)

## 5. Full background pipeline (optional, slow)

Screens, deep-dives candidates, writes a markdown report to `data/reports/`,
and persists the run to SQLite.

In [ ]:
from pipeline_core.services import run_store

run = await orch.execute(PipelineRequest(
    filters=ScreenFilters(index='NIFTY 50', pe_ratio_max=25.0, roe_min=0.15),
    max_candidates=3, include_report=True, analysis_depth='standard',
))
print('status:', run.status, '| candidates:', run.candidates_found, '| reports:', run.report_files)
for r in run.results:
    print(f'  {r.ticker:10s} composite={r.composite_score}')

history = await run_store.list_runs(limit=3)
print('recent runs:', [(h.id[:8], h.status.value) for h in history])